# METAR GeoParquet Interactive Visualization
---

## Overview
   
Within this notebook, we will create interactive visualizations of METAR data at a single hour. We will use the following libraries for our visualizations:

1. [Geopandas](https://geopandas.org/en/stable/)
1. [Lonboard](https://developmentseed.org/lonboard/latest/)

## Prerequisites
| Concepts | Importance | Notes |
| --- | --- | --- |
| [Cartopy Intro](https://foundations.projectpythia.org/core/cartopy/cartopy.html) | Required | Projections and Features |
| [Pandas](https://foundations.projectpythia.org/core/pandas.html) | Required | Tabular Datasets |

- **Time to learn**: 20 minutes
---

## Imports

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
#from cartopy import crs as ccrs
#from cartopy import feature as cfeature
from datetime import datetime,timedelta
#from dateutil.relativedelta import relativedelta

import geopandas as gpd
from lonboard import viz, Map, ScatterplotLayer, HeatmapLayer
import duckdb

### By default, set the date and time to one hour prior to the current time. Or, specify a past date and hour.

In [ ]:
# Use the current time, or set your own for a past time.
# Set current to False if you want to specify a past time.

nowTime = datetime.now()

current = True
current = False
if (current):
    validTime = datetime.now()
    year = validTime.year
    month = validTime.month
    day = validTime.day
    hour = validTime.hour
    validTime = datetime(year, month, day, hour)
    offset = timedelta(hours = 1)
    validTime = validTime - offset
else:
    year = 2026
    month = 1
    day = 1
    hour = 0
    time_1 = datetime(year, month, day, hour)

The timestamps of many hourly METAR reports are 5-10 minutes prior to the top of the hour. When we make our DuckDB query, we will specify a beginning and end time, one hour apart.

In [ ]:
time_0 = time_1 - timedelta(hours=1)
YYYY_0 = time_0.strftime("%Y")
YYYY_1 = time_1.strftime("%Y")
print(time_0, time_1)
# Handle edge case when the two hours straddle the end/beginning of a yearw
if (YYYY_0 == YYYY_1):
    URLs = [f'https://data.source.coop/dynamical/asos-parquet/year={YYYY_1}/data.parquet']
else:
    URLs = [f'https://data.source.coop/dynamical/asos-parquet/year={YYYY_0}/data.parquet',
            f'https://data.source.coop/dynamical/asos-parquet/year={YYYY_1}/data.parquet']


In [ ]:
URLs

### Open the GeoParquet file for the specific year.

In [ ]:
df = duckdb.execute("""
    SELECT *
    FROM read_parquet($1, hive_partitioning=true)
    WHERE 
---      country = 'FR' AND
    valid BETWEEN $2 AND $3
    ORDER BY country
""", [URLs, time_0, time_1]).fetchdf()

In [ ]:
df

In [ ]:
gdf = gpd.GeoDataFrame(df,geometry=gpd.points_from_xy(df.longitude,df.latitude))

In [ ]:
gdf.explore()

In [ ]:
gdf.set_crs(epsg=4326, inplace=True, allow_override=True)

In [ ]:
gdf.explore()

In [ ]:
m = sub1.explore(color='purple')

In [ ]:
# Add title
import folium

title_html = """
<h3 align="center" style="font-size:20px">
    <b>Temperatures >= 100 ° F, 6/24/2024</b>
</h3>
"""

m.get_root().html.add_child(folium.Element(title_html))

# Save or display
m.save("map.html")

In [ ]:
m

In [ ]:
gdf.explore(column='tmpc')

<div class="alert alert-info"><b>Note: </b> Most of the non-US data is not available until approximately 15 minutes past each hour. If you do not see many worldwide stations, try re-reading the data file after that point in the hour.</div>

<div class="alert alert-warning"><b>Exercise: </b> Create a color-coded map from one variable in the dataset.</div>

In [ ]:
# Write your code here
gdfWorldMetar.explore(column='TMPC')

<div class="alert alert-info">Note that the color scale is unaffected by any location whose temperature value was missing ... GeoPandas "knows" to exlucde <code>NaN</code> from the range of valid values.</div>

Plot hourly precipitation

In [ ]:
gdfWorldMetar.explore(column='P01M')

<div class="alert alert-info">With few exceptions, hourly precip is only provided by US stations. For this variable, a better visualization would exclude all the missing values (in the US, a missing value denotes <b>no precip</b>, while a <b>trace</b> = 0.00)</div>

In [ ]:
gdfWorldMetar = gdfWorldMetar.dropna(subset=['P01M']).reset_index(drop=True)

In [ ]:
gdfWorldMetar.explore(column='P01M')

## References

1. [GeoPandas](https://geopandas.org)
1. [EPSG](https://epsg.io)